In [16]:
import pandas as pd

In [25]:
!pip install transformers

In [17]:
!pip install sumy
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
after_sentiment_data = pd.read_csv('/content/processed_newsgroups_dataset_after_sentiment.csv')
before_sentiment_data = pd.read_csv('/content/processed_newsgroups_dataset_before_sentiment.csv')

In [19]:
# Choose a language
LANGUAGE = "english"

# Get a sample text from the 'before_sentiment_data' DataFrame
sample_text = before_sentiment_data['summary'].iloc[0]

# Initialize parser and tokenizer
parser = PlaintextParser.from_string(sample_text, Tokenizer(LANGUAGE))

# Initialize stemmer and summarizer
stemmer = Stemmer(LANGUAGE)
synthesizer = LsaSummarizer(stemmer)
synthesizer.stop_words = get_stop_words(LANGUAGE)

# Generate a summary with 3 sentences
summary_sentences = synthesizer(parser.document, sentences_count=3)

print("Original Text:\n", sample_text)
print("\nExtractive Summary (LSA):\n")
for sentence in summary_sentences:
    print(sentence)

Original Text:
 I am sure some bashers of Pens fans are pretty confused about the lack of any kind of posts about the recent Pens massacre of the Devils. Actually, I am bit puzzled too and a bit relieved. However, I am going to put an end to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they are killing those Devils worse than I thought. Jagr just showed you why he is much better than his regular season stats. He is also a lot fo fun to watch in the playoffs. Bowman should let JAgr have a lot of fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final regular season game. PENS RULE!!!

Extractive Summary (LSA):

However, I am going to put an end to non-PIttsburghers' relief with a bit of praise for the Pens.
Bowman should let JAgr have a lot of fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway.
I was very disappointed no

In [23]:
import pandas as pd

def extract_lsa_summary(text, sentences_count=3, language="english"):
    """
    Generates an extractive summary of the given text using the LSA algorithm.

    Args:
        text (str): The input text to summarize.
        sentences_count (int): The desired number of sentences in the summary.
        language (str): The language of the text (e.g., "english").

    Returns:
        str: The summarized text.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    parser = PlaintextParser.from_string(text, Tokenizer(language))

    actual_num_sentences = len(parser.document.sentences)
    total_words_in_document = len(parser.document.words)

    effective_sentences_count = min(sentences_count, actual_num_sentences)

    if effective_sentences_count > total_words_in_document:
        effective_sentences_count = total_words_in_document

    if effective_sentences_count <= 0:
        return ""

    stemmer = Stemmer(language)
    summarizer = LsaSummarizer(stemmer)
    summarizer.stop_words = get_stop_words(language)

    summary_sentences = summarizer(parser.document, sentences_count=effective_sentences_count)
    return " ".join([str(sentence) for sentence in summary_sentences])

before_sentiment_data['lsa_summary'] = before_sentiment_data['summary'].apply(extract_lsa_summary)

display(before_sentiment_data[['summary', 'lsa_summary']].head())

/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (1) is lower than number of sentences (2). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (1) is lower than number of sentences (3). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (2) is lower than number of sentences (4). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (2) is lower than number of sentences (3). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words

,summary,lsa_summary
0,I am sure some bashers of Pens fans are pretty...,"However, I am going to put an end to non-PItts..."
1,My brother is in the market for a high-perform...,My brother is in the market for a high-perform...
2,Finally you said what you dream about. Mediter...,"The area will be ""greater"" after some years, l..."
3,Think! It's the SCSI card doing the DMA transf...,An important feature of SCSI is the ability to...
4,1) I have an old Jasmine drive which I cannot ...,My understanding is that I have to upsate the ...


In [24]:
before_sentiment_data.to_csv("/content/lsa_summary.csv")

In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

device = torch.device("cpu")
model.to(device)
model.eval()

before_sentiment_data['summary'] = before_sentiment_data['summary'].fillna('').astype(str)

texts = ["summarize: " + t for t in before_sentiment_data['summary'].tolist()]

batch_size = 64

results = []

total_iters = (len(texts) + batch_size - 1) // batch_size

with torch.no_grad():
    for i in range(0, len(texts), batch_size):

        iteration = (i // batch_size) + 1
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            max_length=128,   # 🔥 HUGE speed gain
            truncation=True,
            padding=True
        )

        summary_ids = model.generate(
            inputs['input_ids'],
            max_length=20,    # 🔥 very short summaries
            min_length=5,
            num_beams=1,      # fastest
            do_sample=False
        )

        summaries = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
        results.extend(summaries)

before_sentiment_data['abstractive_summary'] = results

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

1/295
2/295
3/295
4/295
5/295
6/295
7/295
8/295
9/295
10/295
11/295
12/295
13/295
14/295
15/295
16/295
17/295
18/295
19/295
20/295
21/295
22/295
23/295
24/295
25/295
26/295
27/295
28/295
29/295
30/295
31/295
32/295
33/295
34/295
35/295
36/295
37/295
38/295
39/295
40/295
41/295
42/295
43/295
44/295
45/295
46/295
47/295
48/295
49/295
50/295
51/295
52/295
53/295
54/295
55/295
56/295
57/295
58/295
59/295
60/295
61/295
62/295
63/295
64/295
65/295
66/295
67/295
68/295
69/295
70/295
71/295
72/295
73/295
74/295
75/295
76/295
77/295
78/295
79/295
80/295
81/295
82/295
83/295
84/295
85/295
86/295
87/295
88/295
89/295
90/295
91/295
92/295
93/295
94/295
95/295
96/295
97/295
98/295
99/295
100/295
101/295
102/295
103/295
104/295
105/295
106/295
107/295
108/295
109/295
110/295
111/295
112/295
113/295
114/295
115/295
116/295
117/295
118/295
119/295
120/295
121/295
122/295
123/295
124/295
125/295
126/295
127/295
128/295
129/295
130/295
131/295
132/295
133/295
134/295
135/295
136/295
137/295
138/295
139/

In [33]:
before_sentiment_data.to_csv("/content/abstractive_summary.csv")

In [34]:
before_sentiment_data

,text,summary,category_name,lda_categorical_topic,lda_topic_name,lsa_summary,abstractive_summary
0,sure bashers pen fan pretty confused lack kind...,summarize: summarize: I am sure some bashers o...,"rec, sport, hockey",23,"game, team, entry, year, player","However, I am going to put an end to non-PItts...",the pens are killing the Devils worse than I t...
1,brother market high performance video card sup...,summarize: summarize: My brother is in the mar...,"comp, sys, ibm, pc, hardware",29,"window, card, problem, driver, color",My brother is in the market for a high-perform...,a video card supports VESA local bus with 1-2M...
2,finally said dream mediterranean new area grea...,summarize: summarize: Finally you said what yo...,"talk, politics, mideast",27,"armenian, year, turkish, muslim, people","The area will be ""greater"" after some years, l...","the area will be ""greater"" after some years. i..."
3,think scsi card dma transfer disk scsi card dm...,summarize: summarize: Think! It's the SCSI car...,"comp, sys, ibm, pc, hardware",14,"drive, disk, scsi, hard, controller",An important feature of SCSI is the ability to...,the SCSI card can do DMA transfers containing ...
4,old jasmine drive use new system understanding...,summarize: summarize: 1) I have an old Jasmine...,"comp, sys, mac, hardware",14,"drive, disk, scsi, hard, controller",My understanding is that I have to upsate the ...,the drive is a jasmine drive that I cannot use...
...,...,...,...,...,...,...,...
18841,nyeda cnsvax uwec edu david nye neurology cons...,summarize: summarize: DN> From: nyeda@cnsvax.u...,"sci, med",1,"bike, myers, right, dog, time","And also better, because a neurologist can mak...",neurologist can make a differential diagnosis ...
18842,isolated ground recepticles usually unusual co...,summarize: summarize: Not in isolated ground r...,"sci, electronics",3,"power, use, apple, cable, wire",Not in isolated ground recepticles (usually an...,not in isolated ground recepticles (usually an...
18843,installed cpu clone motherboard tried mounting...,summarize: summarize: I just installed a DX2-6...,"comp, sys, ibm, pc, hardware",3,"power, use, apple, cable, wire","After about 1/2 hour, the weight of the cooler...",the CPU fan was enough to dislodge the CPU fro...
18844,wouldn require hyper sphere space point specif...,summarize: summarize: Wouldn't this require a ...,"comp, graphics",12,"think, people, like, thing, point",Wouldn't this require a hyper-sphere. In 3-spa...,steve. steve. steve.. ste
